# QLoRA Fine-tuning(Qwen2.5-3B-Instruct)

This notebook fine-tunes a small language model (SLM) as a complement to the existing RAG pipeline.

**Steps**:
1. Install dependencies (Unsloth + QLoRA)
2. Upload `train.jsonl` and `val.jsonl`
3. Load the base model in 4-bit
4. LoRA configuration
5. Dataset preparation
6. Training
7. Quick before/after test
8. Export 

## 1. Installation


In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir "git+https://github.com/unslothai/unsloth.git"
!pip install trl==0.12.0 peft accelerate bitsandbytes

## 2. Upload your train.jsonl / val.jsonl files

In [ ]:
from google.colab import files
print("Select train.jsonl and val.jsonl:")
uploaded = files.upload()

## 3.Load the base model in 4-bit (QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

## 4. LoRA configuration

In [ ]:
# r=16: good capacity/overfitting-risk trade-off for ~250 examples.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,       # 0 = optimized by Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",  # saves VRAM on long contexts
    random_state=42,
)

## 5. Dataset preparation

In [ ]:
from datasets import load_dataset

def formatting_func(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = load_dataset(
    "json",
    data_files={"train": "train.jsonl", "validation": "val.jsonl"},
)
dataset = dataset.map(formatting_func)

print(dataset)
print("\n--- Formatted example ---\n")
print(dataset["train"][0]["text"])

## 6. Training

- `num_train_epochs=3`: reasonable for ~250 examples without overfitting.
- `learning_rate=2e-4`: standard QLoRA value.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=15,
        save_strategy="no",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()

## 7. Quick before / after test

In [ ]:
FastLanguageModel.for_inference(model)  # enables Unsloth's fast inference mode

test_question = dataset["validation"][0]["messages"][1]["content"]
print("QUESTION:", test_question)
print()

messages = [
    {"role": "system", "content": dataset["validation"][0]["messages"][0]["content"]},
    {"role": "user", "content": test_question},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.3, do_sample=True)
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("FINE-TUNED MODEL RESPONSE:")
print(response)
print()
print("EXPECTED RESPONSE (reference):")
print(dataset["validation"][0]["messages"][2]["content"])

## 8. Export

Three useful formats:
1. **LoRA adapter alone** to reload later with `FastLanguageModel`.
2. **Merged 16-bit model**  to serve directly with vLLM/HuggingFace.
3. **GGUF** to serve locally with **Ollama**.

In [ ]:
# 8a. Save the LoRA adapter only
model.save_pretrained("sfm_qlora_adapter")
tokenizer.save_pretrained("sfm_qlora_adapter")

# 8b. Merge and save the full model in 16-bit
model.save_pretrained_merged("sfm_qwen2.5-3b_merged", tokenizer, save_method="merged_16bit")

# 8c. Export to GGUF
model.save_pretrained_gguf("sfm_qwen2.5-3b_gguf", tokenizer, quantization_method="q4_k_m")

print("Export complete. Download the sfm_qwen2.5-3b_gguf/ folder then:")
print("  1. Create an Ollama Modelfile pointing to the .gguf")
print("  2. ollama create sfm-assistant -f Modelfile")
print("  3. ollama run sfm-assistant")